In [1]:
import os
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, MinMaxScaler
import matplotlib.pyplot as plt
import pickle

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from sklearn.preprocessing import OneHotEncoder
from sklearn.inspection import permutation_importance

from sklearn.semi_supervised import LabelSpreading

In [2]:
# Establish connection with Snowflake
conn = snowflake.connector.connect(connection_name="fundingsociety.ap-southeast-1.privatelink")

In [7]:
# Get SQL query file as data input 
def get_sql_file_as_text(file_path):
  
  with open(file_path, 'r') as f:
    sql_content = f.read()
  return sql_content

sql_file = "for regression.sql"
sql_text = get_sql_file_as_text(sql_file)

# Write SQL Snowflake Query here
my_query=sql_text

# Get data from query above
results = conn.cursor().execute(my_query).fetch_pandas_all()

# Check data
results.head()

,COMPANY_ID,INDUSTRY,VERIFICATION_METHOD,TRANSACTED_GST_BEFORE,TOTAL_TRANSACTIONS_VALUE,TOTAL_TRANSACTIONS_VALUE_SUPPLIER,TOTAL_TRANSACTIONS_VALUE_RENT,TOTAL_TRANSACTIONS_VALUE_PAYROLL,TOTAL_TRANSACTIONS_COUNT,TOTAL_TRANSACTIONS_COUNT_SUPPLIER,TOTAL_TRANSACTIONS_COUNT_RENT,TOTAL_TRANSACTIONS_COUNT_PAYROLL
0,1708,Professional Training & Coaching,Manual,0,568372.016587200000,568372.016587200000,None,None,629,629,0,0
1,458,None,Manual,0,119085.241371600000,50764.266012600000,41429.996048000000,None,73,32,24,0
2,1868,Civil Engineering,Manual,0,675695.967231600000,2650.659440800000,27014.610418600000,646030.697372200000,63,4,10,49
3,1390,None,Manual,0,3796959.630759400000,3796959.630759400000,None,None,1502,1502,0,0
4,1512,None,Manual,0,16282.790206200000,16282.790206200000,None,None,31,31,0,0


In [8]:
# Set index
results = results.set_index('COMPANY_ID')
results.head()

,INDUSTRY,VERIFICATION_METHOD,TRANSACTED_GST_BEFORE,TOTAL_TRANSACTIONS_VALUE,TOTAL_TRANSACTIONS_VALUE_SUPPLIER,TOTAL_TRANSACTIONS_VALUE_RENT,TOTAL_TRANSACTIONS_VALUE_PAYROLL,TOTAL_TRANSACTIONS_COUNT,TOTAL_TRANSACTIONS_COUNT_SUPPLIER,TOTAL_TRANSACTIONS_COUNT_RENT,TOTAL_TRANSACTIONS_COUNT_PAYROLL
COMPANY_ID,,,,,,,,,,,
1708,Professional Training & Coaching,Manual,0,568372.016587200000,568372.016587200000,None,None,629,629,0,0
458,None,Manual,0,119085.241371600000,50764.266012600000,41429.996048000000,None,73,32,24,0
1868,Civil Engineering,Manual,0,675695.967231600000,2650.659440800000,27014.610418600000,646030.697372200000,63,4,10,49
1390,None,Manual,0,3796959.630759400000,3796959.630759400000,None,None,1502,1502,0,0
1512,None,Manual,0,16282.790206200000,16282.790206200000,None,None,31,31,0,0


In [20]:
# Create Dummy Variables
results_with_dummies = pd.get_dummies(results[['INDUSTRY', 'VERIFICATION_METHOD']], dtype=int)

# Combine Dummy data with full data
results_combined = pd.concat([results.drop(['INDUSTRY', 'VERIFICATION_METHOD'], axis=1), results_with_dummies], axis=1)
results_combined.head()

,TRANSACTED_GST_BEFORE,TOTAL_TRANSACTIONS_VALUE,TOTAL_TRANSACTIONS_VALUE_SUPPLIER,TOTAL_TRANSACTIONS_VALUE_RENT,TOTAL_TRANSACTIONS_VALUE_PAYROLL,TOTAL_TRANSACTIONS_COUNT,TOTAL_TRANSACTIONS_COUNT_SUPPLIER,TOTAL_TRANSACTIONS_COUNT_RENT,TOTAL_TRANSACTIONS_COUNT_PAYROLL,INDUSTRY_Accounting,...,INDUSTRY_Translation & Localization,INDUSTRY_Transportation/Trucking/Railroad,INDUSTRY_Venture Capital & Private Equity,INDUSTRY_Veterinary,INDUSTRY_Warehousing,INDUSTRY_Wholesale,INDUSTRY_Wine & Spirits,INDUSTRY_Wireless,VERIFICATION_METHOD_Manual,VERIFICATION_METHOD_MyInfo business
COMPANY_ID,,,,,,,,,,,,,,,,,,,,,
1708,0,568372.016587200000,568372.016587200000,None,None,629,629,0,0,0,...,0,0,0,0,0,0,0,0,1,0
458,0,119085.241371600000,50764.266012600000,41429.996048000000,None,73,32,24,0,0,...,0,0,0,0,0,0,0,0,1,0
1868,0,675695.967231600000,2650.659440800000,27014.610418600000,646030.697372200000,63,4,10,49,0,...,0,0,0,0,0,0,0,0,1,0
1390,0,3796959.630759400000,3796959.630759400000,None,None,1502,1502,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1512,0,16282.790206200000,16282.790206200000,None,None,31,31,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [21]:
# Split Xy data

X = results_combined.drop('TRANSACTED_GST_BEFORE', axis=1)  # Replace 'target_column' with your target variable
y = results_combined['TRANSACTED_GST_BEFORE']

In [25]:
# Scale X using minmax
scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
X_scaled.head()

,TOTAL_TRANSACTIONS_VALUE,TOTAL_TRANSACTIONS_VALUE_SUPPLIER,TOTAL_TRANSACTIONS_VALUE_RENT,TOTAL_TRANSACTIONS_VALUE_PAYROLL,TOTAL_TRANSACTIONS_COUNT,TOTAL_TRANSACTIONS_COUNT_SUPPLIER,TOTAL_TRANSACTIONS_COUNT_RENT,TOTAL_TRANSACTIONS_COUNT_PAYROLL,INDUSTRY_Accounting,INDUSTRY_Airlines/Aviation,...,INDUSTRY_Translation & Localization,INDUSTRY_Transportation/Trucking/Railroad,INDUSTRY_Venture Capital & Private Equity,INDUSTRY_Veterinary,INDUSTRY_Warehousing,INDUSTRY_Wholesale,INDUSTRY_Wine & Spirits,INDUSTRY_Wireless,VERIFICATION_METHOD_Manual,VERIFICATION_METHOD_MyInfo business
COMPANY_ID,,,,,,,,,,,,,,,,,,,,,
1708,0.002850,0.002850,NaN,NaN,0.217452,0.217722,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
458,0.000597,0.000255,0.006563,NaN,0.024931,0.011076,0.036810,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1868,0.003388,0.000013,0.004279,0.095931,0.021468,0.001385,0.015337,0.160131,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1390,0.019039,0.019039,NaN,NaN,0.519737,0.519903,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1512,0.000082,0.000082,NaN,NaN,0.010388,0.010730,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [27]:
# Replace NaN in X to 0
X = X.fillna(0)

In [28]:
# Rebalance using SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [29]:
# Split data to train test
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

In [30]:
model = LogisticRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.84      0.90       856
           1       0.86      0.96      0.91       825

    accuracy                           0.90      1681
   macro avg       0.91      0.90      0.90      1681
weighted avg       0.91      0.90      0.90      1681



/Users/shilton.salindeho/Unmanaged/.conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [32]:
# Calculate permutation importance
permimp = permutation_importance(model, X_test, y_test, random_state=42, n_repeats=30)

# Print results
importances = permimp.importances_mean
indices = permimp.importances_mean.argsort()[-5:]

print("Feature importance:")
for i in indices:
    print(f"{X.columns[i]:20}  {importances[i]:.3f}")

Feature importance:
TOTAL_TRANSACTIONS_VALUE_PAYROLL  0.118
TOTAL_TRANSACTIONS_VALUE_RENT  0.155
TOTAL_TRANSACTIONS_COUNT_SUPPLIER  0.158
TOTAL_TRANSACTIONS_VALUE_SUPPLIER  0.233
TOTAL_TRANSACTIONS_VALUE  0.458


In [39]:
# Get data for no GST only, then try to predict
results_combined_noGSTonly = results_combined[results_combined['TRANSACTED_GST_BEFORE']==0]
X_results_combined_noGSTonly = results_combined_noGSTonly.drop('TRANSACTED_GST_BEFORE', axis=1)

X_results_combined_noGSTonly = X_results_combined_noGSTonly.fillna(0)

y_pred_results_combined_noGSTonly = pd.DataFrame(model.predict(X_results_combined_noGSTonly))
y_pred_results_combined_noGSTonly


,0
0,0
1,1
2,0
3,0
4,0
...,...
4196,0
4197,0
4198,0
4199,0


## Try do Semi-supervised Learning

In [41]:
# Fillna
results_combined = results_combined.fillna(0)

In [43]:
# Transform into labeled vs unlabeled based on GST variable
results_combined_labeled = results_combined[results_combined['TRANSACTED_GST_BEFORE']==1]
results_combined_unlabeled = results_combined[results_combined['TRANSACTED_GST_BEFORE']==0]

X_ss_labeled = results_combined_labeled.drop('TRANSACTED_GST_BEFORE', axis=1)
y_ss_labeled = results_combined_labeled['TRANSACTED_GST_BEFORE']

X_ss_unlabeled = results_combined_unlabeled.drop('TRANSACTED_GST_BEFORE', axis=1)

In [67]:
# Fit semisupervised learning algo to the labeled, then predict the unlabeled
model_ss = LabelSpreading()
model_ss.fit(X_ss_labeled, y_ss_labeled)

y_ss_unlabeled_pred = model.predict(X_ss_unlabeled)

In [66]:
# Concat prediction to whole data
results_combined_unlabeled_withpred = pd.concat([results_combined_unlabeled, pd.DataFrame(y_ss_unlabeled_pred, index=results_combined_unlabeled.index, columns=['predictedGST'])], axis=1)
results_combined_unlabeled_withpred.head()

,TRANSACTED_GST_BEFORE,TOTAL_TRANSACTIONS_VALUE,TOTAL_TRANSACTIONS_VALUE_SUPPLIER,TOTAL_TRANSACTIONS_VALUE_RENT,TOTAL_TRANSACTIONS_VALUE_PAYROLL,TOTAL_TRANSACTIONS_COUNT,TOTAL_TRANSACTIONS_COUNT_SUPPLIER,TOTAL_TRANSACTIONS_COUNT_RENT,TOTAL_TRANSACTIONS_COUNT_PAYROLL,INDUSTRY_Accounting,...,INDUSTRY_Transportation/Trucking/Railroad,INDUSTRY_Venture Capital & Private Equity,INDUSTRY_Veterinary,INDUSTRY_Warehousing,INDUSTRY_Wholesale,INDUSTRY_Wine & Spirits,INDUSTRY_Wireless,VERIFICATION_METHOD_Manual,VERIFICATION_METHOD_MyInfo business,predictedGST
COMPANY_ID,,,,,,,,,,,,,,,,,,,,,
1708,0,568372.016587200000,568372.016587200000,0,0,629,629,0,0,0,...,0,0,0,0,0,0,0,1,0,0
458,0,119085.241371600000,50764.266012600000,41429.996048000000,0,73,32,24,0,0,...,0,0,0,0,0,0,0,1,0,1
1868,0,675695.967231600000,2650.659440800000,27014.610418600000,646030.697372200000,63,4,10,49,0,...,0,0,0,0,0,0,0,1,0,0
1390,0,3796959.630759400000,3796959.630759400000,0,0,1502,1502,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1512,0,16282.790206200000,16282.790206200000,0,0,31,31,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [69]:
# Export to CSV

results_combined_unlabeled_withpred.to_csv('GST predictions.csv')

In [70]:
# Check feature importance
permimp_ls = permutation_importance(model, X_ss_unlabeled, y_ss_unlabeled_pred, random_state=42, n_repeats=30)
importances = permimp_ls.importances_mean
indices = permimp_ls.importances_mean.argsort()[-5:]

print("Feature importance:")
for i in indices:
    print(f"{X_ss_unlabeled.columns[i]:20}  {importances[i]:.3f}")

Feature importance:
TOTAL_TRANSACTIONS_COUNT  0.217
TOTAL_TRANSACTIONS_COUNT_SUPPLIER  0.225
TOTAL_TRANSACTIONS_VALUE_RENT  0.253
TOTAL_TRANSACTIONS_VALUE_SUPPLIER  0.422
TOTAL_TRANSACTIONS_VALUE  0.461
